# Object detection (IoU (Intersection over Union), YOLO (You Only Look Once))

_Deep Learning_

**Not just 'is there a cat?' but 'where is it?' — draw a box and score how well it fits.**

Classification asks "what is in the image?". Object detection also asks "where?", drawing a bounding box around each object.

     To score a predicted box against the true box, we use Intersection-over-Union (IoU): how much they overlap.

---

This notebook is a practice scaffold for the **Object detection (IoU (Intersection over Union), YOLO (You Only Look Once))** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — NumPy

In [ ]:
import numpy as np

def iou(a, b):
    # boxes as (x1, y1, x2, y2)
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)   # overlap area
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter                 # combined area
    return inter / union

truth = (10, 10, 50, 50)        # ground-truth box
print("perfect :", iou(truth, (10, 10, 50, 50)))    # 1.0
print("good    :", round(iou(truth, (15, 15, 55, 55)), 3))
print("no overlap:", iou(truth, (100, 100, 140, 140)))  # 0.0

## Visualize the data & results

_How well does each predicted box overlap the real bounding box of a handwritten digit?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0]                   # real 8x8 image of a 0

ys, xs = np.nonzero(img > 2)             # bright-pixel bounding box
truth = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))

def iou(a, b):                           # boxes as (x1, y1, x2, y2)
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua

preds = {"perfect": truth,
         "tight": tuple(t + 1 for t in truth),
         "loose": (truth[0]-1, truth[1]-1, truth[2]+2, truth[3]+2),
         "shifted": tuple(t + 3 for t in truth),
         "no overlap": (truth[2]+2, truth[3]+2, truth[2]+5, truth[3]+5)}

labels = list(preds)
values = [iou(truth, b) for b in preds.values()]
colors = ["#7ee787", "#7ee787", "#ffb454", "#ff7b72", "#ff7b72"]
plt.bar(labels, values, color=colors)
plt.title("IoU of predicted boxes vs the real digit's bounding box")
plt.show()